In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# NBA Fantasy Basketball Analysis

Three analyses:
1. **VOR Rankings** — all players ranked by Value Over Replacement (overall + by position)
2. **Random Snake Draft** — simulate a random draft and report VOR by pick slot
3. **Fantasy Season Simulation** — bootstrap player game logs to project season standings

In [14]:
from pathlib import Path
import pandas as pd

from fantasy_basketball.player import Player
from fantasy_basketball.scoring import ScoringRules
from fantasy_basketball.vor import LeagueSettings, VORCalculator, compute_base_value
from fantasy_basketball.analysis import generate_vor_rankings, random_snake_draft, simulate_fantasy_season

## Load Data

Reads all 2024-25 game logs from the local cache (no API calls). Then instantiates each `Player` to pull their positions from `sleeper_data.json`.

In [15]:
SEASON = "2024-25"
ASSETS = Path("src/fantasy_basketball/assets") / SEASON

# Load all cached CSVs — no network calls
game_logs = {}
for csv in sorted(ASSETS.glob("*.csv")):
    df = pd.read_csv(csv)
    if not df.empty:
        game_logs[csv.stem] = df

print(f"Loaded {len(game_logs)} players")

Loaded 566 players


In [16]:
# Pull positions from Sleeper cache for every player we have a game log for
positions = {}
skipped = []
for name in game_logs:
    try:
        positions[name] = Player(name).positions
    except ValueError:
        positions[name] = []  # retired mid-season or name mismatch
        skipped.append(name)

print(f"Resolved positions for {len(positions) - len(skipped)} players")
if skipped:
    print(f"Skipped {len(skipped)} players not in active roster: {skipped[:5]}{'...' if len(skipped) > 5 else ''}")

Resolved positions for 566 players


In [17]:
# League configuration — edit these to match your league
rules = ScoringRules.boydfriends()

league = LeagueSettings(
    num_teams=8,
    roster_spots={"G": 2, "F": 2, "C": 1},  # 5 starters per team
    scoring_rules=rules,
    season_games=82,
    risk_aversion=0.1,   # penalizes injured/inconsistent players
)

calc = VORCalculator(league)

In [18]:
# Pre-compute base values and VOR — shared by all three analyses
base_values = {
    name: compute_base_value(log, rules, season_games=82, risk_aversion=0.1)
    for name, log in game_logs.items()
}

replacement_values = calc.compute_replacement_values(base_values, positions)
vor_values = calc.compute_vor(base_values, replacement_values, positions)

print("Replacement values by position:")
for pos, val in replacement_values.items():
    print(f"  {pos}: {val:.2f}")

Replacement values by position:
  G: 19.87
  F: 20.40
  C: 24.05


---
## 1. VOR Rankings

All players ranked by Value Over Replacement — position-agnostic overall rank plus rank within their primary position (G / F / C).

In [19]:
rankings = generate_vor_rankings(game_logs, positions, league)

print(f"Total players ranked: {len(rankings)}\n")
print("=== Top 30 Overall ===")
rankings.head(30)

Total players ranked: 566

=== Top 30 Overall ===


,overall_rank,position_rank,player,primary_position,positions,base_value,vor
0,1,1,Nikola Jokić,C,C,37.9065,13.8560
1,2,1,Shai Gilgeous-Alexander,G,G,31.0783,11.2049
2,3,2,Giannis Antetokounmpo,C,C/F,31.1791,10.7790
3,4,3,Karl-Anthony Towns,C,C/F,27.8644,7.4643
4,5,4,Domantas Sabonis,C,C/F,27.8040,7.4039
5,6,2,James Harden,G,G,27.0291,7.1557
6,7,1,Jayson Tatum,F,F,26.6589,6.2588
7,8,5,Alperen Sengun,C,C/F,26.6010,6.2009
8,9,6,LeBron James,C,C/F,26.0240,5.6239
9,10,2,Josh Hart,F,F/G,25.4963,5.6229


In [20]:
# Top 10 per position
for pos in ["G", "F", "C"]:
    print(f"\n=== Top 10 {pos}s ===")
    top = rankings[rankings["primary_position"] == pos].head(10)
    print(top[["position_rank", "player", "positions", "base_value", "vor"]].to_string(index=False))


=== Top 10 Gs ===
 position_rank                  player positions  base_value     vor
             1 Shai Gilgeous-Alexander         G     31.0783 11.2049
             2            James Harden         G     27.0291  7.1557
             3         Cade Cunningham         G     25.4959  5.6225
             4         Anthony Edwards         G     24.9555  5.0821
             5              Trae Young         G     24.5812  4.7078
             6       Tyrese Haliburton         G     23.9677  4.0943
             7           Dyson Daniels       G/F     23.8344  3.9610
             8            Devin Booker         G     22.5481  2.6747
             9             Tyler Herro         G     22.3512  2.4778
            10             Josh Giddey         G     21.9529  2.0795

=== Top 10 Fs ===
 position_rank             player positions  base_value     vor
             1       Jayson Tatum         F     26.6589  6.2588
             2          Josh Hart       F/G     25.4963  5.6229
           

In [21]:
# Multi-position eligible players (e.g. G/F)
print("=== Dual-Position Eligible Players (Top 20) ===")
rankings[rankings["positions"].str.contains("/")].head(20)

=== Dual-Position Eligible Players (Top 20) ===


,overall_rank,position_rank,player,primary_position,positions,base_value,vor
2,3,2,Giannis Antetokounmpo,C,C/F,31.1791,10.7790
3,4,3,Karl-Anthony Towns,C,C/F,27.8644,7.4643
4,5,4,Domantas Sabonis,C,C/F,27.8040,7.4039
7,8,5,Alperen Sengun,C,C/F,26.6010,6.2009
8,9,6,LeBron James,C,C/F,26.0240,5.6239
9,10,2,Josh Hart,F,F/G,25.4963,5.6229
11,12,7,Bam Adebayo,C,C/F,25.9730,5.5729
16,17,7,Dyson Daniels,G,G/F,23.8344,3.9610
17,18,9,Evan Mobley,C,C/F,23.6524,3.2523
20,21,10,Pascal Siakam,C,C/F,22.6277,2.2276


---
## 2. Random Snake Draft

Simulates an 8-team snake draft where each pick is chosen uniformly at random. Shows which pick slots capture the most VOR on average.

In [22]:
draft = random_snake_draft(vor_values, positions, league, num_rounds=5)

print("=== Full Draft Board ===")
draft

=== Full Draft Board ===


,pick,round,team,player,primary_position,vor
0,1,1,0,Nikola Jokić,C,13.8560
1,2,1,1,Shai Gilgeous-Alexander,G,11.2049
2,3,1,2,Giannis Antetokounmpo,C,10.7790
3,4,1,3,Karl-Anthony Towns,C,7.4643
4,5,1,4,Domantas Sabonis,C,7.4039
5,6,1,5,James Harden,G,7.1557
6,7,1,6,Jayson Tatum,F,6.2588
7,8,1,7,Alperen Sengun,C,6.2009
8,9,2,7,LeBron James,C,5.6239
9,10,2,6,Josh Hart,F,5.6229


In [23]:
# Total VOR captured per team
print("=== VOR by Team ===")
team_vor = draft.groupby("team")["vor"].sum().sort_values(ascending=False).reset_index()
team_vor.columns = ["team", "total_vor"]
print(team_vor.to_string(index=False))

print("\n=== Average VOR by Pick Slot ===")
pick_vor = draft.groupby("pick")["vor"].mean().round(2).reset_index()
pick_vor.columns = ["pick", "avg_vor"]
print(pick_vor.to_string(index=False))

=== VOR by Team ===
 team  total_vor
    0    21.9113
    1    18.9750
    2    18.4996
    4    15.9360
    5    15.6742
    3    15.4782
    6    14.2782
    7    13.8317

=== Average VOR by Pick Slot ===
 pick  avg_vor
    1    13.86
    2    11.20
    3    10.78
    4     7.46
    5     7.40
    6     7.16
    7     6.26
    8     6.20
    9     5.62
   10     5.62
   11     5.62
   12     5.57
   13     5.08
   14     4.71
   15     4.54
   16     4.09
   17     3.96
   18     3.25
   19     2.67
   20     2.48
   21     2.23
   22     2.08
   23     1.64
   24     1.20
   25     1.17
   26     1.09
   27     1.01
   28     0.91
   29     0.51
   30     0.37
   31     0.00
   32     0.00
   33     0.00
   34    -0.02
   35    -0.03
   36    -0.06
   37    -0.17
   38    -0.20
   39    -0.33
   40    -0.36


---
## 3. Fantasy Season Simulation

Uses a greedy VOR draft to assign rosters, then bootstrap-samples each player's historical game log to project 18 weeks of fantasy scores. Each week a team scores above the median earns a win.

In [24]:
# Greedy VOR draft to assign rosters
draft_results = calc.simulate_snake_draft(
    vor_values, positions, num_rounds=5, base_values=base_values
)

print("=== Draft Results ===")
for i, roster in enumerate(draft_results):
    print(f"Team {i}: {', '.join(roster)}")

=== Draft Results ===
Team 0: Nikola Jokić, Dyson Daniels, Tyrese Haliburton, Onyeka Okongwu, Amen Thompson
Team 1: Shai Gilgeous-Alexander, Trae Young, Evan Mobley, Victor Wembanyama, DeMar DeRozan
Team 2: Giannis Antetokounmpo, Anthony Edwards, Pascal Siakam, Josh Giddey, Kevin Durant
Team 3: Karl-Anthony Towns, Cade Cunningham, Nikola Vučević, Jalen Williams, Stephen Curry
Team 4: Domantas Sabonis, Josh Hart, Jarrett Allen, Scottie Barnes, Desmond Bane
Team 5: James Harden, Bam Adebayo, Jalen Duren, Anthony Davis, Austin Reaves
Team 6: Jayson Tatum, LeBron James, Devin Booker, Jaren Jackson Jr., Donovan Mitchell
Team 7: Alperen Sengun, Ivica Zubac, Tyler Herro, Rudy Gobert, Derrick White


In [25]:
standings = simulate_fantasy_season(
    draft_results, game_logs, league,
    n_weeks=18,
    games_per_week=3,
    seed=42,
)

print("=== Projected Season Standings ===")
standings

=== Projected Season Standings ===


,season_rank,team,total_pts,avg_weekly_pts,best_week,worst_week,wins
0,1,1,8129.55,451.64,560.70,397.35,12
1,2,0,8128.70,451.59,490.05,392.80,14
2,3,3,7945.45,441.41,491.85,394.40,12
3,4,2,7753.55,430.75,506.10,351.25,10
4,5,5,7747.90,430.44,477.30,328.05,12
5,6,4,7438.60,413.26,479.10,350.45,5
6,7,6,7280.30,404.46,475.35,338.55,3
7,8,7,7170.40,398.36,475.25,331.00,4


In [26]:
# Combine standings with rosters for full context
print("=== Standings with Rosters ===")
for _, row in standings.iterrows():
    team_idx = int(row["team"])
    roster = ", ".join(draft_results[team_idx])
    print(f"#{int(row['season_rank'])}  Team {team_idx}  {row['total_pts']:.0f}pts  {int(row['wins'])}W  |  {roster}")

=== Standings with Rosters ===
#1  Team 1  8130pts  12W  |  Shai Gilgeous-Alexander, Trae Young, Evan Mobley, Victor Wembanyama, DeMar DeRozan
#2  Team 0  8129pts  14W  |  Nikola Jokić, Dyson Daniels, Tyrese Haliburton, Onyeka Okongwu, Amen Thompson
#3  Team 3  7945pts  12W  |  Karl-Anthony Towns, Cade Cunningham, Nikola Vučević, Jalen Williams, Stephen Curry
#4  Team 2  7754pts  10W  |  Giannis Antetokounmpo, Anthony Edwards, Pascal Siakam, Josh Giddey, Kevin Durant
#5  Team 5  7748pts  12W  |  James Harden, Bam Adebayo, Jalen Duren, Anthony Davis, Austin Reaves
#6  Team 4  7439pts  5W  |  Domantas Sabonis, Josh Hart, Jarrett Allen, Scottie Barnes, Desmond Bane
#7  Team 6  7280pts  3W  |  Jayson Tatum, LeBron James, Devin Booker, Jaren Jackson Jr., Donovan Mitchell
#8  Team 7  7170pts  4W  |  Alperen Sengun, Ivica Zubac, Tyler Herro, Rudy Gobert, Derrick White
